# 01 — Rekordbox Library Exploration

**Research question:** What does my library look like numerically, and does the behavioral data (PlayCount, Rating, playlist membership) contain enough signal to validate future clustering?

**Source:** `data/raw/master.db` — Rekordbox SQLite database  
**Genre context:** EBM, break, electro, IDM  
**Output:** `data/processed/rekordbox_tracks.csv`, `data/processed/playlist_membership.csv`

---

**Findings** *(fill in after running)*
- Total tracks:
- % never played:
- % rated:
- BPM range (10th–90th percentile):
- Playlist co-occurrence signal looks: [strong / weak / uneven]
- Anomalies noticed:

## 0. Imports & config

In [1]:
import sqlite3
import logging
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- config ---
DB_PATH = Path('../data/raw/master.db')
PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR = Path('../outputs/figures')

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# --- logging ---
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# --- plot style ---
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120

logger.info(f'DB path exists: {DB_PATH.exists()}')

INFO: DB path exists: False


## 1. Connect & inspect schema

In [ ]:
conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn
)

print('Available tables:')
print(tables.to_string(index=False))

In [ ]:
# Inspect columns in main content table
# Adjust table name if your Rekordbox version differs
content_info = pd.read_sql_query(
    "PRAGMA table_info(djmdContent);",
    conn
)
print('djmdContent columns:')
print(content_info[['name', 'type']].to_string(index=False))

## 2. Load tracks

In [ ]:
TRACKS_QUERY = """
SELECT
    ID,
    Title,
    Artist,
    BPM,
    Tonality          AS key_raw,
    Genre,
    Label,
    Rating,
    PlayCount,
    DateAdded,
    DateLastPlayed,
    FolderPath,
    FileSize,
    BitRate,
    Duration
FROM djmdContent
WHERE FolderPath IS NOT NULL
"""

df = pd.read_sql_query(TRACKS_QUERY, conn)

logger.info(f'Loaded {len(df):,} tracks')
df.head(3)

## 3. Data quality audit

Know what you have before drawing any conclusions.

In [ ]:
print('=== Shape ===')
print(f'Tracks: {len(df):,}  |  Columns: {df.shape[1]}')

print('\n=== Missing values ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
audit = pd.DataFrame({'missing': missing, '%': missing_pct})
print(audit[audit['missing'] > 0])

print('\n=== Data types ===')
print(df.dtypes)

In [ ]:
# Behavioral signal coverage — how much signal do we actually have?
never_played = (df['PlayCount'] == 0).sum()
rated = (df['Rating'] > 0).sum()
has_bpm = df['BPM'].notna().sum()
has_key = df['key_raw'].notna().sum()

print('=== Signal coverage ===')
print(f"Never played:     {never_played:,} ({never_played/len(df)*100:.1f}%)")
print(f"Has rating:       {rated:,} ({rated/len(df)*100:.1f}%)")
print(f"Has BPM:          {has_bpm:,} ({has_bpm/len(df)*100:.1f}%)")
print(f"Has key:          {has_key:,} ({has_key/len(df)*100:.1f}%)")

# NOTE: PlayCount only counts plays within Rekordbox — not actual DJ performance
# High never-played % is normal if you import tracks before auditioning them

## 4. Clean & derive base features

In [ ]:
df_clean = df.copy()

# --- key mapping ---
# Rekordbox stores key as integer 0-23 (0-11 major, 12-23 minor)
# Adjust if your version uses different encoding
KEY_MAP = {
    0: 'C',  1: 'Db', 2: 'D',  3: 'Eb', 4: 'E',  5: 'F',
    6: 'Gb', 7: 'G',  8: 'Ab', 9: 'A',  10: 'Bb', 11: 'B',
    12: 'Cm', 13: 'Dbm', 14: 'Dm', 15: 'Ebm', 16: 'Em', 17: 'Fm',
    18: 'Gbm', 19: 'Gm', 20: 'Abm', 21: 'Am', 22: 'Bbm', 23: 'Bm'
}
df_clean['key'] = df_clean['key_raw'].map(KEY_MAP)

# --- dates ---
df_clean['DateAdded'] = pd.to_datetime(df_clean['DateAdded'], unit='s', errors='coerce')
df_clean['DateLastPlayed'] = pd.to_datetime(df_clean['DateLastPlayed'], unit='s', errors='coerce')

today = pd.Timestamp.now()
df_clean['days_in_library'] = (today - df_clean['DateAdded']).dt.days
df_clean['days_since_played'] = (today - df_clean['DateLastPlayed']).dt.days

# --- behavioral derived features ---
df_clean['plays_per_day'] = (
    df_clean['PlayCount'] / (df_clean['days_in_library'] + 1)
)
df_clean['is_rated'] = (df_clean['Rating'] > 0).astype(int)
df_clean['is_played'] = (df_clean['PlayCount'] > 0).astype(int)

# --- fill / drop ---
df_clean['Rating'] = df_clean['Rating'].fillna(0)
df_clean['Genre'] = df_clean['Genre'].fillna('Unknown')
df_clean['Label'] = df_clean['Label'].fillna('Unknown')
df_clean = df_clean.dropna(subset=['BPM', 'FolderPath'])

logger.info(f'Clean dataset: {len(df_clean):,} tracks')

## 5. Playlist data & co-occurrence

In [ ]:
PLAYLIST_QUERY = """
SELECT
    sp.ContentID  AS track_id,
    p.Name        AS playlist_name,
    p.ID          AS playlist_id
FROM djmdSongPlaylist sp
JOIN djmdPlaylist p ON sp.PlaylistID = p.ID
WHERE p.Name IS NOT NULL
"""

df_playlists = pd.read_sql_query(PLAYLIST_QUERY, conn)
logger.info(f'Playlist entries: {len(df_playlists):,}')

# Playlist co-occurrence: how many playlists each track appears in
playlist_counts = (
    df_playlists.groupby('track_id')['playlist_id']
    .nunique()
    .reset_index()
    .rename(columns={'playlist_id': 'playlist_count'})
)

df_clean = df_clean.merge(
    playlist_counts,
    left_on='ID',
    right_on='track_id',
    how='left'
)
df_clean['playlist_count'] = df_clean['playlist_count'].fillna(0).astype(int)

print(f"Tracks in at least 1 playlist: {(df_clean['playlist_count'] > 0).sum():,}")
print(f"Max playlist appearances: {df_clean['playlist_count'].max()}")
print('\nPlaylist count distribution:')
print(df_clean['playlist_count'].value_counts().sort_index().head(10))

In [ ]:
# Preference score — combines behavioral signals
# Formula from Topic 2 decisions: log(playcount)*0.5 + rating*0.4 + playlist*0.1
max_playlist = df_clean['playlist_count'].max() or 1

df_clean['preference_score'] = (
    np.log1p(df_clean['PlayCount']) * 0.5 +
    (df_clean['Rating'] / 5.0) * 0.4 +
    (df_clean['playlist_count'] / max_playlist) * 0.1
)

print('Preference score stats:')
print(df_clean['preference_score'].describe().round(3))

print('\nTop 10 highest preference score tracks:')
top = df_clean.nlargest(10, 'preference_score')[[
    'Artist', 'Title', 'BPM', 'PlayCount', 'Rating', 'playlist_count', 'preference_score'
]]
print(top.to_string(index=False))

## 6. Exploratory statistics

In [ ]:
print('=== BPM ===')
print(df_clean['BPM'].describe().round(1))
print(f"10th–90th pct: {df_clean['BPM'].quantile(0.1):.1f} – {df_clean['BPM'].quantile(0.9):.1f}")

print('\n=== PlayCount ===')
print(df_clean['PlayCount'].describe().round(1))

print('\n=== Rating distribution ===')
print(df_clean['Rating'].value_counts().sort_index())

print('\n=== Top 10 labels ===')
print(df_clean['Label'].value_counts().head(10))

print('\n=== Most played tracks ===')
most_played = df_clean.nlargest(10, 'PlayCount')[['Artist', 'Title', 'BPM', 'PlayCount']]
print(most_played.to_string(index=False))

## 7. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Rekordbox Library Overview', fontsize=15, fontweight='bold')

# 1. BPM distribution
axes[0, 0].hist(df_clean['BPM'], bins=60, color='steelblue', edgecolor='none', alpha=0.85)
axes[0, 0].set_title('BPM Distribution')
axes[0, 0].set_xlabel('BPM')
axes[0, 0].set_ylabel('Tracks')

# 2. PlayCount (log scale — expected heavy skew)
played = df_clean[df_clean['PlayCount'] > 0]['PlayCount']
axes[0, 1].hist(played, bins=50, color='coral', edgecolor='none', alpha=0.85)
axes[0, 1].set_yscale('log')
axes[0, 1].set_title('Play Count Distribution (played tracks, log scale)')
axes[0, 1].set_xlabel('Play Count')

# 3. Rating
rating_counts = df_clean[df_clean['Rating'] > 0]['Rating'].value_counts().sort_index()
axes[0, 2].bar(rating_counts.index, rating_counts.values, color='mediumseagreen', edgecolor='none')
axes[0, 2].set_title('Rating Distribution (rated tracks only)')
axes[0, 2].set_xlabel('Rating')
axes[0, 2].set_ylabel('Tracks')

# 4. Preference score distribution
axes[1, 0].hist(df_clean['preference_score'], bins=50, color='mediumpurple', edgecolor='none', alpha=0.85)
axes[1, 0].set_title('Preference Score Distribution')
axes[1, 0].set_xlabel('Score')

# 5. BPM vs preference score — does taste correlate with tempo?
sc = axes[1, 1].scatter(
    df_clean['BPM'],
    df_clean['preference_score'],
    alpha=0.3,
    s=15,
    c=df_clean['PlayCount'],
    cmap='plasma'
)
fig.colorbar(sc, ax=axes[1, 1], label='PlayCount')
axes[1, 1].set_title('BPM vs Preference Score')
axes[1, 1].set_xlabel('BPM')
axes[1, 1].set_ylabel('Preference Score')

# 6. Playlist co-occurrence
axes[1, 2].hist(df_clean['playlist_count'], bins=30, color='darkorange', edgecolor='none', alpha=0.85)
axes[1, 2].set_title('Playlist Co-occurrence')
axes[1, 2].set_xlabel('Number of playlists track appears in')
axes[1, 2].set_ylabel('Tracks')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'library_overview.png', dpi=150, bbox_inches='tight')
plt.show()
logger.info('Saved: library_overview.png')

In [ ]:
# Top labels by average preference score
# More informative than count — which labels do you actually play?
label_stats = (
    df_clean[df_clean['Label'] != 'Unknown']
    .groupby('Label')
    .agg(
        track_count=('ID', 'count'),
        avg_preference=('preference_score', 'mean'),
        avg_plays=('PlayCount', 'mean')
    )
    .query('track_count >= 3')  # ignore labels with 1-2 tracks
    .sort_values('avg_preference', ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(label_stats.index, label_stats['avg_preference'], color='steelblue', alpha=0.85)
ax.set_title('Top Labels by Average Preference Score (min 3 tracks)')
ax.set_xlabel('Avg Preference Score')
ax.invert_yaxis()

# Annotate with track count
for bar, count in zip(bars, label_stats['track_count']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'n={count}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'labels_by_preference.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Cold start decision

Tracks with PlayCount=0 are 'cold' — no behavioral signal. Decide how to handle them before Sprint 3.

Options:
- **Exclude from training** — cleaner signal, smaller dataset
- **Include as candidates only** — train on played tracks, surface unplayed as discovery
- **Include with preference_score=0** — noisier but preserves audio features

**Decision made:** *(fill in)*

In [ ]:
cold = df_clean[df_clean['PlayCount'] == 0]
warm = df_clean[df_clean['PlayCount'] > 0]

print(f'Cold tracks (never played): {len(cold):,} ({len(cold)/len(df_clean)*100:.1f}%)')
print(f'Warm tracks (played):       {len(warm):,} ({len(warm)/len(df_clean)*100:.1f}%)')
print(f'\nCold track BPM range: {cold["BPM"].quantile(0.1):.1f} – {cold["BPM"].quantile(0.9):.1f}')
print(f'Warm track BPM range: {warm["BPM"].quantile(0.1):.1f} – {warm["BPM"].quantile(0.9):.1f}')

# If BPM distributions differ significantly, cold tracks may skew your feature space

## 9. History data (if available)

In [ ]:
# Check if djmdHistory exists and has useful timestamps
# Not all Rekordbox versions populate this table

try:
    history_info = pd.read_sql_query("PRAGMA table_info(djmdHistory);", conn)
    print('djmdHistory columns:')
    print(history_info[['name', 'type']].to_string(index=False))

    df_history = pd.read_sql_query("SELECT * FROM djmdHistory LIMIT 5;", conn)
    print(f'\nSample rows:')
    print(df_history)

except Exception as e:
    logger.warning(f'History table not available or empty: {e}')
    print('djmdHistory unavailable — recency_weighted_plays feature will be skipped')

## 10. Save outputs

In [ ]:
# Save cleaned track dataset
df_clean.to_csv(PROCESSED_DIR / 'rekordbox_tracks.csv', index=False)
logger.info(f'Saved rekordbox_tracks.csv — {len(df_clean):,} tracks')

# Save playlist membership (used in Sprint 3 + 4 for cluster validation)
df_playlists.to_csv(PROCESSED_DIR / 'playlist_membership.csv', index=False)
logger.info(f'Saved playlist_membership.csv — {len(df_playlists):,} entries')

# Close DB connection
conn.close()
logger.info('DB connection closed')

print('\n=== Sprint 1 complete ===')
print(f'  Tracks exported: {len(df_clean):,}')
print(f'  Features ready for Sprint 2: BPM, key, Rating, PlayCount, preference_score, playlist_count')
print(f'  Cold start decision needed before Sprint 3')